
### JAB-Hessian sensitivity estimation & adaptive precision allocation



**Contents**
1. GPTQ core (shared quantizer, same as Student A's)
2. Calibration data capture (X and target attention output A(X))
3. Attention-aware joint loss (MSE + optional KL)
4. Hutchinson trace estimator
5. Greedy sensitivity-per-cost allocator
6. Shared utilities (calibration batches, perplexity)
7. Load GPT-2 + validate the from-scratch attention loss against the real model
8. JAB-Hessian: per-block sensitivity scores
9. Full pipeline: uniform GPTQ baseline
10. Full pipeline: JAB-Hessian adaptive allocation
11. Compare results


In [1]:
!pip install -q torch transformers datasets

In [2]:
import math
import random
import itertools

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd.functional import hessian as exact_hessian
import torch.autograd as autograd

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

Using device: cuda


## 1. GPTQ core


In [3]:
def collect_hessian_via_hook(model: torch.nn.Module, module: torch.nn.Module,
                              calibration_batches, device) -> torch.Tensor:
    """
    Registers a forward pre-hook on `module` (e.g. one block's attn.c_attn)
    to capture its input activations, runs `calibration_batches` through the
    *whole model* in no_grad mode, and returns the accumulated Hessian
    H = 2 X^T X for that layer.

    `calibration_batches` should be an iterable of input_ids tensors of shape
    (batch, seq_len), already on `device`.

    Returns: H, a (d_in, d_in) double-precision tensor.
    """
    d_in = module.weight.shape[0]  # Conv1D weight is (in_features, out_features)
    H = torch.zeros(d_in, d_in, dtype=torch.float64, device=device)
    n_samples = [0]

    def _hook(mod, inputs):
        x = inputs[0].detach()
        x = x.reshape(-1, x.shape[-1]).to(torch.float64)  # (tokens, d_in)
        H.add_(2.0 * x.T @ x)
        n_samples[0] += x.shape[0]

    handle = module.register_forward_pre_hook(_hook)
    try:
        model.eval()
        with torch.no_grad():
            for input_ids in calibration_batches:
                model(input_ids.to(device))
    finally:
        handle.remove()

    if n_samples[0] > 0:
        H /= n_samples[0]
    return H


def _quantize_to_grid(w_col: torch.Tensor, scale: torch.Tensor, bits: int) -> torch.Tensor:
    """
    Symmetric per-output-row fake quantization of a single input-column
    (shape: d_out) using a fixed per-row scale (shape: d_out) computed
    up front from the original weight statistics.
    """
    qmax = 2 ** (bits - 1) - 1
    q = torch.clamp(torch.round(w_col / scale), -qmax, qmax)
    return q * scale


@torch.no_grad()
def gptq_quantize_layer(weight_in_out: torch.Tensor, H: torch.Tensor, bits: int = 4,
                         damp_percent: float = 0.01, group_size: int = None,
                         act_order: bool = True, return_scale: bool = False) -> torch.Tensor:
    """
    Quantizes a weight matrix in the (d_in, d_out) "Conv1D" convention
    (GPT-2 style: forward is x @ weight) using the GPTQ algorithm.

    weight_in_out: (d_in, d_out) float tensor -- e.g. c_attn.weight.data
    H: (d_in, d_in) Hessian from collect_hessian_via_hook
    bits: target bit-width
    damp_percent: Hessian damping factor for numerical stability (GPTQ default ~0.01)
    group_size: if set (e.g. 128), computes a separate per-row scale for each
        contiguous group of `group_size` input columns instead of one scale
        for the whole row. Finer granularity -> lower quantization error,
        at essentially no extra cost. None = one scale per row (previous
        default behavior).
    act_order: if True, quantizes columns in order of decreasing Hessian
        diagonal (most "sensitive" columns first, while the most
        compensation budget is still available) instead of naive
        left-to-right order. Standard GPTQ accuracy improvement.

    return_scale: if True, ALSO returns the exact per-(output-channel,
        group) scale tensor that was used, expressed in the ORIGINAL
        (unpermuted) column order and in the same (d_in, d_out) orientation
        as weight_in_out -- so a caller can later fake-quantize the SAME
        weight with the SAME grid (e.g. for STE fine-tuning warm-starts)
        and get back exactly what GPTQ produced, instead of silently
        re-quantizing onto a different, coarser grid.

    Returns the fake-quantized weight, same shape, and also writes it into
    weight_in_out in place. If return_scale, returns (W_final, scale) instead.
    """
    device = weight_in_out.device
    W = weight_in_out.detach().clone().to(torch.float64).T.contiguous()  # (d_out, d_in)
    d_out, d_in = W.shape

    # Damp and (optionally) reorder the Hessian before inverting.
    H = H.clone()
    mean_diag = H.diagonal().mean()
    H += damp_percent * mean_diag * torch.eye(d_in, dtype=torch.float64, device=device)

    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        invperm = torch.argsort(perm)
        W = W[:, perm]
        H = H[perm][:, perm]
    else:
        perm = torch.arange(d_in, device=device)
        invperm = perm

    H_inv = torch.linalg.inv(H)

    # Per-(row, group) scale, computed once from the original weights
    # (in the possibly-permuted column order) before any quantization.
    qmax = 2 ** (bits - 1) - 1
    gs = group_size if group_size is not None else d_in
    n_groups = (d_in + gs - 1) // gs
    scale = torch.zeros(d_out, n_groups, dtype=torch.float64, device=device)
    for g in range(n_groups):
        start, end = g * gs, min((g + 1) * gs, d_in)
        scale[:, g] = (W[:, start:end].abs().amax(dim=1) / qmax).clamp(min=1e-8)

    for i in range(d_in):
        row_scale = scale[:, i // gs]
        w_col = W[:, i]
        q_col = _quantize_to_grid(w_col, row_scale, bits)
        err = (w_col - q_col) / H_inv[i, i]
        if i + 1 < d_in:
            W[:, i + 1:] -= torch.outer(err, H_inv[i, i + 1:])
        W[:, i] = q_col

    if act_order:
        W = W[:, invperm]

    W_final = W.T.contiguous().to(weight_in_out.dtype)  # back to (d_in, d_out)
    weight_in_out.copy_(W_final)

    if not return_scale:
        return W_final

    # Expand the per-(row, group) scale to a per-(row, column) scale in the
    # SAME permuted column order used above, then undo act_order's
    # permutation so it lines up with weight_in_out's original columns.
    group_idx_per_permuted_col = torch.arange(d_in, device=device) // gs   # (d_in,)
    scale_per_permuted_col = scale[:, group_idx_per_permuted_col]           # (d_out, d_in)
    scale_per_original_col = scale_per_permuted_col[:, invperm] if act_order else scale_per_permuted_col
    scale_final = scale_per_original_col.T.contiguous().to(weight_in_out.dtype)  # (d_in, d_out)
    return W_final, scale_final

## 2. Calibration data capture


In [4]:
class AttentionOutputCapture:
    """
    Captures attention output A(X) from each block, used for MSE loss.
    """
    def __init__(self, model, detach=True):
        self.outputs = {}
        self.handles = []
        self.detach = detach

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_proj (this is A(X) before output projection)
            handle = block.attn.c_proj.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, inputs):
            # input[0] is A(X) - the attention output
            if self.detach:
                self.outputs[idx] = inputs[0].detach().clone()
            else:
                self.outputs[idx] = inputs[0]  # ← Keep gradients!
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


class ActivationCapture:
    """
    Captures input activations (X) to each block's attention.
    """
    def __init__(self, model, detach=True):
        self.activations = {}
        self.handles = []
        self.detach = detach

        for idx, block in enumerate(model.transformer.h):
            # Hook before c_attn (this is X, the input to attention)
            handle = block.attn.c_attn.register_forward_pre_hook(self._make_hook(idx))
            self.handles.append(handle)

    def _make_hook(self, idx):
        def hook(module, input):
            # input[0] is X
            if self.detach:
                self.activations[idx] = input[0].detach().clone()
            else:
                self.activations[idx] = input[0]  # ← Keep gradients!
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()


def get_calibration_data(model, calibration_batch, device, with_grad=False):
    """
    Get all calibration data for one forward pass.
    Returns X, target_A, target_attn.

    # ===== CHANGED (KL divergence) =====
    # target_attn used to be hard-coded to None ("MSE only for now"). It is
    # now populated with each block's REAL post-softmax attention-weight
    # matrix, obtained via output_attentions=True on this same forward pass
    # (no extra forward pass needed). This is what kl_loss/attention_loss's
    # target_attn argument expects, so the KL term defined in Section 3 can
    # actually be used instead of always silently falling back to MSE-only.
    # NOTE: requires the model to have been loaded with
    # attn_implementation="eager" (sdpa/flash attention do not expose
    # attention-weight tensors).
    """
    # Create captures
    act_capture = ActivationCapture(model, detach=not with_grad)
    out_capture = AttentionOutputCapture(model, detach=not with_grad)

    # Run model
    model.eval()
    if with_grad:
        # Enable gradients for Fisher coupling
        with torch.enable_grad():
            outputs = model(calibration_batch.to(device), output_attentions=True)
            # Keep gradients for target_attn
            target_attn = {idx: attn for idx, attn in enumerate(outputs.attentions)}
    else:
        # Default: no gradients (for Hessian trace)
        with torch.no_grad():
            outputs = model(calibration_batch.to(device), output_attentions=True)
            target_attn = {idx: attn.detach().clone() for idx, attn in enumerate(outputs.attentions)}

    # Get data
    X = act_capture.activations
    target_A = out_capture.outputs
    # ===== CHANGED: target_attn is now real data, not None =====
    # outputs.attentions: tuple of (batch, n_head, T, T) tensors, one per block,
    # already post-softmax -- same quantity compute_attention returns as attn_weights.
    #target_attn = {idx: attn.detach().clone() for idx, attn in enumerate(outputs.attentions)}

    # Clean up
    act_capture.remove()
    out_capture.remove()

    return X, target_A, target_attn


def get_calibration_data_for_block(model, calibration_batch, device, block_idx, with_grad=False):
    """
    Get calibration data for a specific block.
    """
    X_dict, target_A_dict, target_attn_dict = get_calibration_data(
        model, calibration_batch, device, with_grad=with_grad
    )

    return (
        X_dict[block_idx],
        target_A_dict[block_idx],
        target_attn_dict[block_idx],  # ===== CHANGED: was hard-coded None =====
    )

## 3. Attention-aware joint loss
`L = ||A(X) - A_hat(X)||^2 (+ lambda * KL(attention maps), optional)` -- a from-scratch, differentiable multi-head attention computation (with causal masking) as a pure function of a flattened `[W_Q | W_K | W_V]` parameter vector, which is exactly what the Hutchinson estimator needs to differentiate through.

In [5]:
def reshape_weights(w_flat, n_embd):
    """
    Convert flattened weights back to Q, K, V matrices.
    Inputs:
        w_flat: Flattened [W_Q, W_K, W_V]
        n_embd: Model dimension (768 for GPT-2 small)
    Outputs:
        W_Q, W_K, W_V: Each of shape (n_embd, n_embd)
    """
    # Each matrix has n_embd * n_embd parameters
    size = n_embd * n_embd

    # Split into 3 parts
    W_Q = w_flat[0:size].reshape(n_embd, n_embd)
    W_K = w_flat[size:2*size].reshape(n_embd, n_embd)
    W_V = w_flat[2*size:3*size].reshape(n_embd, n_embd)

    return W_Q, W_K, W_V

def compute_attention(W_Q, W_K, W_V, X, n_head = 12, b_Q=None, b_K=None, b_V=None):
    """
    Compute attention output and attention weights.
    Inputs:
        W_Q, W_K, W_V: Weight matrices (n_embd, n_embd)
        X: Input activations (batch, seq_len, n_embd)
    Outputs:
        A_hat: Attention output (batch, seq_len, n_embd)
        attn_weights: Attention weights (batch, seq_len, seq_len)
    """
    B, T, n_embd = X.shape #batch size, sequence length, 768
    d_head = n_embd // n_head #per-head dimension
    # Compute Q, K, V
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    if b_Q is not None:
        Q = Q + b_Q
        K = K + b_K
        V = V + b_V

    Q = Q.view(B, T, n_head, d_head).transpose(1, 2)
    K = K.view(B, T, n_head, d_head).transpose(1, 2)
    V = V.view(B, T, n_head, d_head).transpose(1, 2)
    # Scaled dot-product attention
    #d_k = Q.shape[-1]
    #scale = torch.sqrt(torch.tensor(d_k, dtype=torch.float32, device=Q.device))
    scale = math.sqrt(d_head)
    scores = Q @ K.transpose(-2, -1) / scale

    causal_mask = torch.tril(torch.ones(T, T, device=X.device, dtype=torch.bool))
    scores = scores.masked_fill(~causal_mask, float("-inf"))
    # Attention weights
    attn_weights = torch.softmax(scores, dim=-1)
    # Attention output (weighted sum of values)
    #A_hat = attn_weights @ V
    A_hat = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, n_embd)

    return A_hat, attn_weights

def mse_loss(w_flat, X, target_A, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_mse = ||A(X) - A_hat(X)||^2
    This is the primary loss function.
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    A_hat, _ = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # MSE loss
    return F.mse_loss(A_hat, target_A)

def kl_loss(w_flat, X, target_attn, n_embd, b_Q=None, b_K=None, b_V=None):
    """
    L_kl = KL(attention_weights || target_attention_weights)
    """
    # Reshape and compute attention
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    _, attn_weights = compute_attention(W_Q, W_K, W_V, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # KL divergence: KL(P || Q) = sum(P * log(P / Q))
    # P = target_attn, Q = attn_weights
    log_q = torch.log(attn_weights + 1e-8)  # Add epsilon for stability

    return F.kl_div(log_q, target_attn, reduction='batchmean') #batchmean = sum(KL) / batch_size (like in Q-BERT & APTQ)

def attention_loss(w_flat, X, target_A, n_embd, target_attn=None, lambda_kl=0.1, b_Q=None, b_K=None, b_V=None):
    """
    If target_attn is None: Returns MSE only O.W. :Returns MSE + lambda_kl * KL
    """
    # Always compute MSE
    loss = mse_loss(w_flat, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # Add KL if target attention weights are provided
    if target_attn is not None:
        kl = kl_loss(w_flat, X, target_attn, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
        loss = loss + lambda_kl * kl

    return loss

### Sanity check: `attention_loss.py` unit tests
All run on synthetic data (no GPT-2 needed) -- verifies shapes, gradient flow, numerical stability, and that MSE-only and MSE+KL agree when KL's weight/target is absent.

In [6]:
def test_reshape_weights():
    print("\n=== Test 1: Reshape Weights ===")
    n_embd = 768
    total = 3 * n_embd * n_embd
    # Create random flat weights
    w_flat = torch.randn(total)
    # Reshape
    W_Q, W_K, W_V = reshape_weights(w_flat, n_embd)
    # Check shapes
    assert W_Q.shape == (n_embd, n_embd)
    assert W_K.shape == (n_embd, n_embd)
    assert W_V.shape == (n_embd, n_embd)
    # Check we can reconstruct
    reconstructed = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])
    assert torch.allclose(w_flat, reconstructed, atol=1e-6)
    print("Reshape works correctly")
    print(f"   Input shape: {w_flat.shape}")
    print(f"   W_Q shape: {W_Q.shape}")

def test_compute_attention():
    print("\n=== Test 2: Compute Attention ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random inputs
    W_Q = torch.randn(n_embd, n_embd)
    W_K = torch.randn(n_embd, n_embd)
    W_V = torch.randn(n_embd, n_embd)
    X = torch.randn(batch, seq, n_embd)
    # Compute attention
    A_hat, attn_weights = compute_attention(W_Q, W_K, W_V, X)
    n_head = 12  # default in compute_attention
    # Check shapes
    assert A_hat.shape == (batch, seq, n_embd)
    assert attn_weights.shape == (batch, n_head, seq, seq)
    # Check attention weights sum to 1 (per row)
    sums = attn_weights.sum(dim=-1)
    assert torch.allclose(sums, torch.ones(batch, n_head, seq), atol=1e-5)
    print("Attention computation works")
    print(f"   A_hat shape: {A_hat.shape}")
    print(f"   Attn weights sum: {sums[0, 0, 0].item():.6f}")

def test_mse_loss():
    print("\n=== Test 3: MSE Loss ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random data
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Compute loss
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Check it's a scalar and non-negative
    assert loss.shape == ()  # Scalar
    assert loss.item() >= 0  # Non-negative
    print(f"MSE loss works")
    print(f"   Loss: {loss.item():.6f}")

def test_gradients():
    print("\n=== Test 4: Gradient Flow ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create data with requires_grad
    w_flat = torch.randn(3 * n_embd * n_embd, requires_grad=True)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Forward pass
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Backward pass
    loss.backward()
    # Check gradients
    assert w_flat.grad is not None
    assert w_flat.grad.shape == w_flat.shape
    assert torch.any(w_flat.grad != 0)  # Non-zero gradients
    print(f"Gradients flow correctly")
    print(f"   Gradient shape: {w_flat.grad.shape}")
    print(f"   Gradient norm: {w_flat.grad.norm().item():.6f}")

def test_kl_loss():
    print("\n=== Test 5: KL Divergence Loss ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Create random data
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Create random target attention weights (valid probability distribution)
    n_head = 12
    target_attn = torch.softmax(torch.randn(batch, n_head, seq, seq), dim=-1)
    # MSE only
    loss_mse = attention_loss(w_flat, X, target_A, n_embd)
    # MSE + KL
    loss_combined = attention_loss(w_flat, X, target_A, n_embd, target_attn, lambda_kl=0.1)
    # Combined loss should be >= MSE loss (since KL >= 0)
    assert loss_combined.item() >= loss_mse.item()
    print(f"KL loss works")
    print(f"   MSE only: {loss_mse.item():.6f}")
    print(f"   MSE + KL: {loss_combined.item():.6f}")

def test_different_shapes():
    print("\n=== Test 6: Different Shapes ===")
    n_embd = 768
    # Test different batch sizes
    for batch in [1, 2, 4]:
        seq = 4
        w_flat = torch.randn(3 * n_embd * n_embd)
        X = torch.randn(batch, seq, n_embd)
        target_A = torch.randn(batch, seq, n_embd)

        loss = attention_loss(w_flat, X, target_A, n_embd)
        print(f"   Batch {batch}: loss = {loss.item():.6f}")
    # Test different sequence lengths
    for seq in [1, 4, 8]:
        batch = 2
        w_flat = torch.randn(3 * n_embd * n_embd)
        X = torch.randn(batch, seq, n_embd)
        target_A = torch.randn(batch, seq, n_embd)

        loss = attention_loss(w_flat, X, target_A, n_embd)
        print(f"   Seq {seq}: loss = {loss.item():.6f}")

    print("Works with different shapes")

def test_numerical_stability():
    print("\n=== Test 7: Numerical Stability ===")
    n_embd = 768
    batch = 2
    seq = 4
    # Test with large values
    w_flat = torch.randn(3 * n_embd * n_embd) * 1000
    X = torch.randn(batch, seq, n_embd) * 1000
    target_A = torch.randn(batch, seq, n_embd) * 1000
    loss = attention_loss(w_flat, X, target_A, n_embd)
    # Should not be NaN or Inf
    assert not torch.isnan(loss)
    assert not torch.isinf(loss)
    print(f"Stable with extreme values: {loss.item():.6f}")

def test_attention_loss_consistency():
    print("\n=== Test 8: Loss Consistency ===")
    n_embd = 768
    batch = 2
    seq = 4
    w_flat = torch.randn(3 * n_embd * n_embd)
    X = torch.randn(batch, seq, n_embd)
    target_A = torch.randn(batch, seq, n_embd)
    # Method 1: Direct mse_loss
    loss1 = mse_loss(w_flat, X, target_A, n_embd)
    # Method 2: attention_loss without KL
    loss2 = attention_loss(w_flat, X, target_A, n_embd)
    # Method 3: attention_loss with target_attn=None
    loss3 = attention_loss(w_flat, X, target_A, n_embd, target_attn=None)
    assert torch.allclose(loss1, loss2, atol=1e-6)
    assert torch.allclose(loss1, loss3, atol=1e-6)
    print(f"All methods give consistent results")
    print(f"   mse_loss: {loss1.item():.6f}")
    print(f"   attention_loss: {loss2.item():.6f}")

if __name__ == "__main__":
    print("TESTING ATTENTION_LOSS.PY (No GPT-2 Required)")

    test_reshape_weights()
    test_compute_attention()
    test_mse_loss()
    test_gradients()
    test_kl_loss()
    test_different_shapes()
    test_numerical_stability()
    test_attention_loss_consistency()
    print("ALL TESTS PASSED!")
    print("\nThe loss function is ready to use with GPT-2.")

TESTING ATTENTION_LOSS.PY (No GPT-2 Required)

=== Test 1: Reshape Weights ===
Reshape works correctly
   Input shape: torch.Size([1769472])
   W_Q shape: torch.Size([768, 768])

=== Test 2: Compute Attention ===
Attention computation works
   A_hat shape: torch.Size([2, 4, 768])
   Attn weights sum: 1.000000

=== Test 3: MSE Loss ===
MSE loss works
   Loss: 766.721497

=== Test 4: Gradient Flow ===
Gradients flow correctly
   Gradient shape: torch.Size([1769472])
   Gradient norm: 25.978100

=== Test 5: KL Divergence Loss ===
KL loss works
   MSE only: 770.092224
   MSE + KL: 831.760620

=== Test 6: Different Shapes ===
   Batch 1: loss = 812.840149
   Batch 2: loss = 762.878967
   Batch 4: loss = 785.917786
   Seq 1: loss = 717.577209
   Seq 4: loss = 730.669922
   Seq 8: loss = 769.112732
Works with different shapes

=== Test 7: Numerical Stability ===
Stable with extreme values: 785353761882112.000000

=== Test 8: Loss Consistency ===
All methods give consistent results
   mse_loss

## 4. Hutchinson trace estimator
`trace(H) ~= (1/n) * sum(v_i^T H v_i)` for Rademacher vectors `v_i`, using two backward passes (double-backward) to get exact Hessian-vector products without ever forming the full Hessian.

In [7]:
def hessian_vector_product(loss_fn, params, vector, retain_graph = True):
    #first order grad
    grad = autograd.grad(loss_fn(params), params, create_graph=True, retain_graph=True)[0] #retain_graph = true to keep the graph for second grad, do not release it!
    #second order grad
    hvp = autograd.grad(grad, params, grad_outputs=vector, retain_graph=True)[0]
    return hvp

def hutchinson_trace_estimator(loss_fn, params, samples=50): #number of iterations mentioned in HAWQ-V2 article
    #trace(H) ≈ (1/n) * Σ(v_i^T * H * v_i)
    if not params.requires_grad:
        params.requires_grad_(True)

    device = params.device
    estimated_trace = 0.0

    for _ in range(samples):
        #Rademacher Vector(mentioned in HAWQ-V2)
        vec = torch.randint(0, 2, params.shape, device=device) * 2 -1
        vec = vec.float()
        hvp_result = hessian_vector_product(loss_fn, params, vec)
        estimated_trace += torch.dot(vec.flatten(), hvp_result.flatten())

    return estimated_trace/samples

### Sanity check: Hutchinson estimate vs. exact Hessian trace (toy layer)
On a tiny `nn.Linear(5, 3)` we can afford to compute the *exact* Hessian via `torch.autograd.functional.hessian` and compare it to the Hutchinson estimate directly.

In [8]:
def test_hutchinson():

    #toy layer
    layer = nn.Linear(5, 3)
    x = torch.randn(4, 5) #input data
    y = torch.randn(4, 3) #target data

    weights = list(layer.parameters())[0].clone()
    weights_flat = weights.flatten()
    weights_flat.requires_grad_(True)
    print(f"shape of weights: {weights.shape}")
    print(f"number of weights: {weights.numel()}")

    def loss_fn(w):
        w_reshaped = w.reshape(3, 5)
        output = torch.matmul(x, w_reshaped.T) + layer.bias
        return nn.MSELoss()(output, y)

    test_loss = loss_fn(weights_flat)
    print(f"Test loss: {test_loss.item():.6f}")

    H_matrix = exact_hessian(loss_fn, weights_flat)
    H_2d = H_matrix.reshape(weights_flat.numel(), weights_flat.numel())
    exact_trace = torch.trace(H_2d)
    print(f"exact trace: {exact_trace:0.6f}")

    hutchinson_trace = hutchinson_trace_estimator(loss_fn=loss_fn, params=weights_flat, samples=300)
    print(f"Hutchinson estimated trace: {hutchinson_trace:0.6f}")

    print("\n3. Comparing results...")
    error = abs(hutchinson_trace - exact_trace)
    print(f"Exact trace:      {exact_trace:0.6f}")
    print(f"Hutchinson:       {hutchinson_trace:0.6f}")
    print(f"Error:            {error:0.6f}")

    if error < 0.1:
        print("Test passed!")
    else:
        print(f"Error too large: {error:0.6f}")

torch.manual_seed(0)  # fixed seed: this is a Monte Carlo estimator,
# without a seed the pass/fail outcome is flaky run-to-run (verified:
# a real toy-problem run gave error 0.05 once and 0.32 another time,
# purely from random-draw variance, not a code issue)
test_hutchinson()

shape of weights: torch.Size([3, 5])
number of weights: 15
Test loss: 1.258623
exact trace: 6.869429
Hutchinson estimated trace: 6.871145

3. Comparing results...
Exact trace:      6.869429
Hutchinson:       6.871145
Error:            0.001717
Test passed!


## 5. Greedy sensitivity-per-cost allocator
Starts every block at the highest bit-width, then repeatedly downgrades whichever block loses the least accuracy per unit of budget freed, until the budget is met -- then spends any leftover budget on the best available upgrades.

In [9]:
#Greedy sensitivity-per-cost bit-width allocator
import random
import itertools

BIT_WIDTHS = [2, 3, 4, 8, 16] #2, 16 is deleted for now!

def generate_synthetic_scores(block_names, seed=0):
    """
    Returns a dictionary of sensitivity scores per bit width per each block.
    """
    rng = random.Random(seed)
    scores = {}

    for name in block_names:
        fragility = rng.uniform(0.5, 3.0)
        sensitivity_by_bits = {}

        for bits in BIT_WIDTHS:
            # more bits -> less sensitivity
            sensitivity = fragility / (bits ** 1.5)
            sensitivity_by_bits[bits] = sensitivity

        scores[name] = sensitivity_by_bits

    return scores

def cost(bits, param_count=1.0):
    """
    Memory cost of storing a block at a given bit-width.
    """
    return bits * param_count

def greedy_allocate(scores, budget):
    """
    scores: dict of {block_name: {bits: sensitivity}}
    budget: max total cost allowed
      1. Give every block the HIGHEST bit-width (best accuracy, most cost).
      2. While we're over budget: find the one downgrade (one block, one
         step down in bits) that saves the most cost per unit of accuracy
         lost, and apply it. Repeat.
      3. If we still have leftover budget afterward, try upgrading blocks
         back up wherever it's affordable and helps the most.
    """
    # start at max precision
    current_bits = {name: max(BIT_WIDTHS) for name in scores}

    def total_cost():
        return sum(cost(current_bits[name]) for name in scores)

    def total_sensitivity():
        return sum(scores[name][current_bits[name]] for name in scores)

    # downgrade loop
    while total_cost() > budget:
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (cost)

        for name in scores:
            bits_now = current_bits[name]
            lower_choices = [b for b in BIT_WIDTHS if b < bits_now]
            if not lower_choices:
                continue  # already at the lowest possible bit-width

            next_bits = max(lower_choices)
            cost_saved = cost(bits_now) - cost(next_bits)
            sensitivity_added = scores[name][next_bits] - scores[name][bits_now]

            ratio = sensitivity_added / cost_saved

            if best_ratio is None or ratio < best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is None:
            break  # can't downgrade anything further

        current_bits[best_block] = best_new_bits

    # spend remaining budget on the best upgrade available
    made_an_upgrade = True
    while made_an_upgrade:
        made_an_upgrade = False
        best_block = None
        best_new_bits = None
        best_ratio = None  # (sensitivity) / (extra cost)

        for name in scores:
            bits_now = current_bits[name]
            higher_choices = [b for b in BIT_WIDTHS if b > bits_now]
            if not higher_choices:
                continue

            next_bits = min(higher_choices)
            extra_cost = cost(next_bits) - cost(bits_now)

            if total_cost() + extra_cost > budget:
                continue  # can't afford it

            sensitivity_saved = scores[name][bits_now] - scores[name][next_bits]
            ratio = sensitivity_saved / extra_cost

            if best_ratio is None or ratio > best_ratio:
                best_ratio = ratio
                best_block = name
                best_new_bits = next_bits

        if best_block is not None:
            current_bits[best_block] = best_new_bits
            made_an_upgrade = True

    return current_bits, total_cost(), total_sensitivity()

def brute_force_optimal(scores, budget):
    """
    Tries every possible combination of bit-widths and keeps the best one that fits the budget. Only usable for a small number of blocks.
    """
    names = list(scores.keys())
    choices_per_block = [BIT_WIDTHS] * len(names)

    best_assignment = None
    best_cost = None
    best_sensitivity = float("inf")

    for combo in itertools.product(*choices_per_block):
        total_c = sum(cost(bits) for bits in combo)
        if total_c > budget:
            continue

        total_s = sum(scores[names[i]][combo[i]] for i in range(len(names)))

        if total_s < best_sensitivity:
            best_sensitivity = total_s
            best_cost = total_c
            best_assignment = dict(zip(names, combo))

    return best_assignment, best_cost, best_sensitivity

### Sanity check: greedy allocator vs. brute-force optimal (synthetic scores)
On a small 4-block slice we can afford to check every combination exactly, to confirm the greedy heuristic isn't leaving accuracy on the table.

In [10]:
# 12 blocks = W_Q, W_K, W_V for 4 layers
block_names = [f"L{layer}_{proj}" for layer in range(4) for proj in ("WQ", "WK", "WV")]
scores = generate_synthetic_scores(block_names, seed=42)

max_cost = sum(max(BIT_WIDTHS) for _ in block_names)
min_cost = sum(min(BIT_WIDTHS) for _ in block_names)
budget = min_cost + 0.35 * (max_cost - min_cost)

print(f"{len(block_names)} blocks | min_cost={min_cost} max_cost={max_cost} budget={budget:.1f}\n")

assignment, cost_used, sensitivity = greedy_allocate(scores, budget)
print("Greedy allocation:")
for name in block_names:
    print(f"  {name:10s} -> {assignment[name]:2d} bits")
print(f"\nTotal cost: {cost_used:.1f} (budget {budget:.1f})")
print(f"Total sensitivity: {sensitivity:.4f}")
# validate against brute force on a small 4-block slice
small_names = block_names[:4]
small_scores = {name: scores[name] for name in small_names}
small_max = sum(max(BIT_WIDTHS) for _ in small_names)
small_min = sum(min(BIT_WIDTHS) for _ in small_names)
small_budget = small_min + 0.35 * (small_max - small_min)

greedy_assign, greedy_cost, greedy_sens = greedy_allocate(small_scores, small_budget)
optimal_assign, optimal_cost, optimal_sens = brute_force_optimal(small_scores, small_budget)

print("\n--- Validation on 4 blocks ---")
print(f"Greedy : cost={greedy_cost:.1f} sensitivity={greedy_sens:.4f} -> {greedy_assign}")
print(f"Optimal: cost={optimal_cost:.1f} sensitivity={optimal_sens:.4f} -> {optimal_assign}")

12 blocks | min_cost=24 max_cost=192 budget=82.8

Greedy allocation:
  L0_WQ      ->  8 bits
  L0_WK      ->  4 bits
  L0_WV      ->  8 bits
  L1_WQ      ->  8 bits
  L1_WK      ->  8 bits
  L1_WV      ->  8 bits
  L2_WQ      ->  8 bits
  L2_WK      ->  4 bits
  L2_WV      ->  8 bits
  L3_WQ      ->  4 bits
  L3_WK      ->  4 bits
  L3_WV      ->  8 bits

Total cost: 80.0 (budget 82.8)
Total sensitivity: 1.0223

--- Validation on 4 blocks ---
Greedy : cost=24.0 sensitivity=0.3478 -> {'L0_WQ': 8, 'L0_WK': 4, 'L0_WV': 8, 'L1_WQ': 4}
Optimal: cost=27.0 sensitivity=0.3002 -> {'L0_WQ': 8, 'L0_WK': 3, 'L0_WV': 8, 'L1_WQ': 8}


## 6. Shared utilities
Calibration-batch construction and sliding-window perplexity evaluation (same as Student A's).

In [11]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

def build_calibration_batches(tokenizer, n_samples=128, seq_len=512):
    """
    Pulls `n_samples` chunks of `seq_len` tokens each from WikiText-2 train,
    as a list of (1, seq_len) input_id tensors.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids[0]

    batches = []
    stride = seq_len
    for i in range(n_samples):
        start = i * stride
        if start + seq_len > ids.shape[0]:
            break
        chunk = ids[start:start + seq_len].unsqueeze(0)
        batches.append(chunk)
    return batches


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, max_length=1024, stride=512):
    """
    Sliding-window perplexity on WikiText-2 test.
    """
    raw = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(t for t in raw["text"] if t.strip())
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    seq_len = ids.shape[1]

    model.eval()
    nll_sum = 0.0
    n_tokens = 0
    prev_end = 0

    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        input_ids = ids[:, begin:end]
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        out = model(input_ids, labels=target_ids)
        nll_sum += out.loss.item() * trg_len
        n_tokens += trg_len

        prev_end = end
        if end == seq_len:
            break

    return math.exp(nll_sum / n_tokens)

## 7. Load GPT-2 and validate the from-scratch attention loss
Before trusting JAB-Hessian scores computed from `attention_loss.py`'s reimplemented attention, confirm it reproduces GPT-2's *real* attention output almost exactly.

In [12]:
# ===== CHANGED: attn_implementation="eager" so output_attentions=True (used
# by get_calibration_data for the KL target) returns real attention weights.
# sdpa/flash-attention backends don't materialize attention-weight tensors. =====
model = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model.eval()
n_embd = model.config.n_embd
print(f"Model dimension: {n_embd}")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Model dimension: 768


In [13]:
# a small real batch
text = "The quick brown fox jumps over the lazy dog. " * 20
batch = tokenizer(text, return_tensors="pt", truncation=True, max_length=64).input_ids.to(DEVICE)

block_idx = 0
# ===== CHANGED: target_attn is no longer discarded (used to be `_`) =====
X, target_A, target_attn = get_calibration_data_for_block(model, batch, DEVICE, block_idx)

# reconstruct w_flat from the REAL, unquantized weights of this block
block = model.transformer.h[block_idx]
W = block.attn.c_attn.weight.data  # (768, 2304) = [W_Q | W_K | W_V]
W_Q, W_K, W_V = W.split(n_embd, dim=1)
w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])

# reconstruct the real bias too -- same split pattern as the weights
bias = block.attn.c_attn.bias.data  # (2304,)
b_Q, b_K, b_V = bias.split(n_embd)

with torch.no_grad():
    err = mse_loss(w_flat, X, target_A, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)
    W_Q_r, W_K_r, W_V_r = reshape_weights(w_flat, n_embd)
    A_hat, attn_weights = compute_attention(W_Q_r, W_K_r, W_V_r, X, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # ===== ADDED: validate target_attn the same way target_A is validated =====
    kl_value = kl_loss(w_flat, X, target_attn, n_embd, b_Q=b_Q, b_K=b_K, b_V=b_V)

mse_value = err.item()
target_scale = (target_A**2).mean().item()
kl_value = kl_value.item()  # ===== ADDED =====

print(f"attn_weights shape (expect (batch, 12, T, T)): {tuple(attn_weights.shape)}")
print(f"target_attn shape (expect same as attn_weights): {tuple(target_attn.shape)}")  # ===== ADDED =====
print(f"MSE between real attention output and compute_attention's output: {mse_value:.10f}")
print(f"Scale reference -- mean squared magnitude of target_A itself: {target_scale:.10f}")
print(f"Ratio (MSE / target scale): {mse_value / target_scale:.10f}")
# ===== ADDED: KL(recomputed attn_weights || real target_attn) should be ~0,
# since compute_attention reconstructs the same real attention this block
# actually produced -- this validates the new target_attn capture path that
# the KL loss term now depends on. =====
print(f"KL(recomputed attn_weights || real target_attn): {kl_value:.10f}")

if mse_value / target_scale < 1e-4:
    print("\nPASS -- compute_attention reconstructs the real attention output almost exactly.")
else:
    print("\nFAIL -- still a meaningful gap; compute_attention does not match GPT-2's real attention yet.")

if kl_value < 1e-4:  # ===== ADDED =====
    print("PASS -- recomputed attention weights match real target_attn (KL term is trustworthy).")
else:
    print("FAIL -- recomputed attention weights diverge from target_attn; check output_attentions wiring.")

attn_weights shape (expect (batch, 12, T, T)): (1, 12, 64, 64)
target_attn shape (expect same as attn_weights): (1, 12, 64, 64)
MSE between real attention output and compute_attention's output: 0.0000000000
Scale reference -- mean squared magnitude of target_A itself: 0.0253827553
Ratio (MSE / target scale): 0.0000000000
KL(recomputed attn_weights || real target_attn): -0.0002380554

PASS -- compute_attention reconstructs the real attention output almost exactly.
PASS -- recomputed attention weights match real target_attn (KL term is trustworthy).


## 8. JAB-Hessian: per-block sensitivity scores
Wires the Hutchinson estimator to the real attention-output loss (Section 3) to get one sensitivity score per block, converts it to a per-bit-width sensitivity table, and runs the greedy allocator (Section 5) against it.

In [14]:
def compute_jab_trace_from_data(model, block_idx, X, target_A, device, n_embd, samples=30,
                                 target_attn=None, lambda_kl=0.1):
    """
    Compute sensitivity score per attention block.
    inputs:
        model: GPT-2 model
        block_idx: Which layer
        batch: One batch of input tokens
        n_embd: Model dimension
        samples: Number of Hutchinson samples
        target_attn: real post-softmax attention weights for this block (KL
            target). If None, this reduces to the previous MSE-only trace.
            # ===== ADDED =====
        lambda_kl: weight on the KL term when target_attn is provided.
            # ===== ADDED =====
    """
    # 1. Get weights of this block's Q, K, V
    block = model.transformer.h[block_idx]
    W = block.attn.c_attn.weight.data  # Shape: (768, 2304) = [W_Q | W_K | W_V]
    # Split into Q, K, V and flatten them into one vector
    W_Q, W_K, W_V = W.split(n_embd, dim=1)
    w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])
    w_flat.requires_grad_(True)  # for Hessian computation
    # 2. Get calibration data (X = input, target_A = attention output)
    bias = block.attn.c_attn.bias.data
    b_Q, b_K, b_V = bias.split(n_embd)
    # 3. Loss function
    def loss_fn(params):
        # ===== CHANGED: now passes target_attn/lambda_kl through, so the
        # Hessian trace (and therefore the sensitivity score / bit
        # allocation) reflects MSE + lambda_kl * KL, not just MSE. =====
        return attention_loss(params, X, target_A, n_embd, target_attn=target_attn,
                               lambda_kl=lambda_kl, b_Q=b_Q, b_K=b_K, b_V=b_V)
    # 4. Compute Hessian trace using Hutchinson estimator
    trace = hutchinson_trace_estimator(loss_fn, w_flat, samples=samples)
    return trace.item()

def compute_all_jab_scores(model, batches, device, n_embd, samples=30, n_batches_to_use=4,
                            lambda_kl=0.1):  # ===== CHANGED: added lambda_kl =====
    """
    Compute sensitivity scores for ALL attention blocks and returns it in format of a dictionary.
    """
    n_blocks = len(model.transformer.h)
    use_batches = batches[:n_batches_to_use]
    all_traces = {f"block_{i}_QKV": [] for i in range(n_blocks)}

    print(f"\nComputing JAB scores for {n_blocks} blocks "
          f"(averaged over {len(use_batches)} batches, {samples} Hutchinson samples each, "
          f"loss = MSE + {lambda_kl}*KL)...")  # ===== CHANGED: message reflects joint loss =====

    for b in use_batches:
        # ONE forward pass captures X/target_A/target_attn for every block at once
        # ===== CHANGED: target_attn_dict used to be discarded with `_` =====
        X_dict, target_A_dict, target_attn_dict = get_calibration_data(model, b, device)
        for idx in range(n_blocks):
            trace = compute_jab_trace_from_data(
                model, idx, X_dict[idx], target_A_dict[idx], device, n_embd, samples,
                target_attn=target_attn_dict[idx], lambda_kl=lambda_kl,  # ===== ADDED =====
            )
            all_traces[f"block_{idx}_QKV"].append(trace)

    scores = {}
    for block_name, traces in all_traces.items():
        trace = sum(traces) / len(traces)
        scores[block_name] = trace
        level = "HIGH" if trace > 100 else "MEDIUM" if trace > 10 else "LOW"
        print(f"  {block_name}: {trace:.4f}  [{level} sensitivity]  "
              f"(min={min(traces):.4f}, max={max(traces):.4f})")

    print("Done!\n")
    return scores

def scores_to_allocator_format(jab_scores):
    """
    Convert JAB scores to format expected by greedy allocator.
    """
    allocator_input = {}
    bit_widths = [2, 3, 4, 8, 16]#[2,

    for block_name, trace in jab_scores.items():
        sensitivity_by_bits = {}
        for bits in bit_widths:
            sensitivity_by_bits[bits] = trace / (bits ** 1.5)
        allocator_input[block_name] = sensitivity_by_bits

    return allocator_input

def measure_weight_perturbation(original_W, H, bits):
    """
    HAWQ-V2 style: ||Q(W) - W||_F^2, measured directly in weight space.
    """
    w_copy = original_W.clone()
    gptq_quantize_layer(w_copy, H, bits=bits, group_size=128, act_order=True)
    perturbation = torch.norm(w_copy - original_W, p='fro') ** 2
    return perturbation.item()


def scores_to_allocator_format_hawqv2(jab_scores, model, calibration_batches, device, bit_widths=None):
    """
    HAWQ-V2 style: Omega_i(bits) = trace_i * ||Q(W_i) - W_i||_F^2,
    instead of the guessed trace / bits^1.5.
    """
    if bit_widths is None:
        bit_widths = [2, 3, 4, 8, 16]

    allocator_input = {}
    for block_name, trace in jab_scores.items():
        block_idx = int(block_name.split("_")[1])          # extract the index from "block_0_QKV" -> 0
        block = model.transformer.h[block_idx]
        c_attn = block.attn.c_attn
        original_W = c_attn.weight.data

        H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)

        sensitivity_by_bits = {}
        for bits in bit_widths:
            perturbation = measure_weight_perturbation(original_W, H, bits)
            sensitivity_by_bits[bits] = trace * perturbation
        allocator_input[block_name] = sensitivity_by_bits

        print(f"  {block_name}: " + ", ".join(f"{b}bit={v:.4e}" for b, v in sensitivity_by_bits.items()))

    return allocator_input


def run_jab_allocation(model, batches, device, n_embd, target_avg_bits=4.0, samples=30,
                        n_batches_to_use=16, lambda_kl=0.1):  # ===== CHANGED: added lambda_kl =====

    """
    1. Compute JAB scores for all blocks
    2. Convert to allocator format (HAWQ-V2 style, measured perturbation)
    3. Run greedy allocation
    4. Return bit assignment in format of a dictionary
    """
    print("JAB-Hessian Adaptive Allocation")
    # Compute JAB scores
    print("\nComputing JAB scores...")
    # ===== CHANGED: lambda_kl threaded through to compute_all_jab_scores =====
    jab_scores = compute_all_jab_scores(model, batches, device, n_embd, samples, n_batches_to_use,
                                         lambda_kl=lambda_kl)
    # Convert format
    print("Preparing for allocator...")
    allocator_input = scores_to_allocator_format_hawqv2(jab_scores, model, batches, device)
    # Set budget
    n_blocks = len(allocator_input)
    budget = target_avg_bits * n_blocks
    print(f"Budget: {budget:.1f} bits ({target_avg_bits} avg for {n_blocks} blocks)")
    # Run allocation
    print("\nRunning greedy allocation...")
    assignment, cost_used, sensitivity = greedy_allocate(allocator_input, budget)
    # results
    print("Results")
    print(f"Total cost: {cost_used:.1f} bits")
    print(f"Average bits: {cost_used / n_blocks:.2f}")
    print(f"Total sensitivity: {sensitivity:.6f}")
    print("\nPer-block allocation:")
    for block_name, bits in assignment.items():
        print(f"    {block_name}: {bits} bits")

    return assignment

## 9. Full pipeline: uniform GPTQ baseline
Quantizes every block's `c_attn` to a flat 4 bits -- the R7 comparison point for adaptive allocation. Both this and Section 10 reuse the *same* calibration batches, so the comparison is apples-to-apples.

**Note (adopted from Untitled10.ipynb's `gptq_core.py`):** like that notebook's `quantize_all_attention_blocks`, each block's Hessian is collected and quantized in sequence on the *same* live model object, so later blocks' calibration activations already reflect earlier blocks' quantization error -- this is not fully independent per-block quantization. The next cell also reuses Untitled10's saved GPTQ checkpoint (`week1_gptq_qkv_init.pt`) when present, instead of always re-running GPTQ from scratch.

In [15]:
import os

print("Building the shared calibration set (used for all experiments below)...")
calibration_batches = build_calibration_batches(tokenizer, n_samples=128, seq_len=512)#32,128
print(f"{len(calibration_batches)} calibration batches ready.\n")

print("Loading a fresh full-precision GPT-2 for the uniform baseline...")
model_uniform = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
model_uniform.eval()

# --- Checkpoint-first: reuse Untitled10.ipynb's saved GPTQ init if present on disk ---
# Untitled10's `save_quantized_qkv` step writes exactly this filename. Loading it here
# instead of re-running GPTQ avoids a redundant multi-minute pass when both notebooks
# are run in the same environment. NOTE: Untitled10's own `gptq_quantize_layer` does
# NOT support `group_size`/`act_order`, so a checkpoint it produced was quantized
# WITHOUT those refinements -- slightly different from what this cell's own GPTQ call
# below would produce. If you want this baseline to reflect this notebook's improved
# quantizer (group_size=128, act_order=True), delete the checkpoint file and let this
# cell regenerate it.
CHECKPOINT_PATH = "week1_gptq_qkv_init.pt"

if os.path.exists(CHECKPOINT_PATH):
    print(f"Found {CHECKPOINT_PATH} -- loading GPTQ-quantized Q/K/V instead of re-running GPTQ...")
    state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    for idx, block in enumerate(model_uniform.transformer.h):
        saved = state[f"block_{idx}"]
        W_cat = torch.cat([saved["W_Q"], saved["W_K"], saved["W_V"]], dim=1).to(DEVICE)
        block.attn.c_attn.weight.data.copy_(W_cat)
    print("Loaded.\n")
else:
    print(f"No {CHECKPOINT_PATH} found -- quantizing ALL blocks to 4 bits uniformly...")
    for idx in range(len(model_uniform.transformer.h)):
        block = model_uniform.transformer.h[idx]
        c_attn = block.attn.c_attn
        print(f"  Block {idx}: quantizing to 4 bits...")
        H = collect_hessian_via_hook(model_uniform, c_attn, calibration_batches, DEVICE)
        gptq_quantize_layer(c_attn.weight.data, H, bits=4, group_size=128, act_order=True)

    # Save so this pass doesn't need to be repeated -- same format as Untitled10's Step 4,
    # so either notebook can load either notebook's checkpoint going forward.
    save_state = {}
    for idx, block in enumerate(model_uniform.transformer.h):
        W_Q, W_K, W_V = block.attn.c_attn.weight.data.split(n_embd, dim=1)
        save_state[f"block_{idx}"] = {
            "W_Q": W_Q.clone().cpu(), "W_K": W_K.clone().cpu(), "W_V": W_V.clone().cpu()
        }
    torch.save(save_state, CHECKPOINT_PATH)
    print(f"Saved GPTQ-initialized Q/K/V to {CHECKPOINT_PATH} for reuse.\n")

print("\nEvaluating perplexity...")
ppl_uniform = evaluate_perplexity(model_uniform, tokenizer)
print(f"\nUniform 4-bit perplexity: {ppl_uniform:.3f}")


Building the shared calibration set (used for all experiments below)...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

...

128 calibration batches ready.

Loading a fresh full-precision GPT-2 for the uniform baseline...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

No week1_gptq_qkv_init.pt found -- quantizing ALL blocks to 4 bits uniformly...
  Block 0: quantizing to 4 bits...
  Block 1: quantizing to 4 bits...
  Block 2: quantizing to 4 bits...
  Block 3: quantizing to 4 bits...
  Block 4: quantizing to 4 bits...
  Block 5: quantizing to 4 bits...
  Block 6: quantizing to 4 bits...
  Block 7: quantizing to 4 bits...
  Block 8: quantizing to 4 bits...
  Block 9: quantizing to 4 bits...
  Block 10: quantizing to 4 bits...
  Block 11: quantizing to 4 bits...
Saved GPTQ-initialized Q/K/V to week1_gptq_qkv_init.pt for reuse.


Evaluating perplexity...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Uniform 4-bit perplexity: 26.225


## 10. Full pipeline: JAB-Hessian adaptive allocation
Computes JAB scores, allocates bits under an average-bit budget close to the uniform baseline, applies it, and evaluates perplexity -- using the *same* calibration batches as Section 9.

**Budget note:** we use `target_avg_bits=4.3`, not `4.0`. At exactly 4.0 the budget lands on a `BIT_WIDTHS` grid point and every block saturates to the same 4-bit floor before any differentiation happens (see the sweep cell right after this one for a full explanation and a demonstration that the allocator differentiates correctly away from that point).

In [16]:
def apply_allocation(model, assignment, calibration_batches, device):
    print("\nApplying allocation")
    for idx in range(len(model.transformer.h)):
        block_name = f"block_{idx}_QKV"
        if block_name in assignment:
            bits = assignment[block_name]
            block = model.transformer.h[idx]
            c_attn = block.attn.c_attn
            H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)
            gptq_quantize_layer(c_attn.weight.data, H, bits=bits, group_size=128, act_order=True)
            print(f"  Block {idx}: {bits} bits")
    print("Allocation done.")
    return model

In [17]:
print("Loading a fresh full-precision GPT-2 for adaptive allocation...")
# ===== CHANGED: attn_implementation="eager" -- this model goes through
# get_calibration_data (inside run_jab_allocation), which now requests
# output_attentions=True for the KL target. =====
model_adaptive = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
model_adaptive.eval()

print("\nComputing JAB-Hessian scores and allocating bits...")
torch.manual_seed(42)
# target_avg_bits=4.3, not 4.0 -- see markdown above (4.0 sits exactly on a BIT_WIDTHS grid point)
# ===== CHANGED: lambda_kl=0.1 makes the allocation reflect MSE + 0.1*KL,
# not MSE alone. Set lambda_kl=0.0 to reproduce the old MSE-only behavior. =====
assignment = run_jab_allocation(model=model_adaptive, batches=calibration_batches,
                                 device=DEVICE, n_embd=n_embd, target_avg_bits=4.3, samples=20,
                                 lambda_kl=0.1)

print("\nApplying the allocation...")
model_adaptive = apply_allocation(model_adaptive, assignment, calibration_batches, DEVICE)

print("\nEvaluating perplexity...")
ppl_adaptive = evaluate_perplexity(model_adaptive, tokenizer)
print(f"\nAdaptive (JAB-Hessian) perplexity: {ppl_adaptive:.3f}")

Loading a fresh full-precision GPT-2 for adaptive allocation...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing JAB-Hessian scores and allocating bits...
JAB-Hessian Adaptive Allocation

Computing JAB scores...

Computing JAB scores for 12 blocks (averaged over 16 batches, 20 Hutchinson samples each, loss = MSE + 0.1*KL)...
  block_0_QKV: 7126.0962  [HIGH sensitivity]  (min=6753.5308, max=7727.5493)
  block_1_QKV: 18893.3922  [HIGH sensitivity]  (min=17823.1191, max=19570.4277)
  block_2_QKV: 109225.9614  [HIGH sensitivity]  (min=99855.3203, max=117951.3359)
  block_3_QKV: 292531.0693  [HIGH sensitivity]  (min=259636.7031, max=313418.2188)
  block_4_QKV: 136848.2666  [HIGH sensitivity]  (min=124662.5859, max=147360.7969)
  block_5_QKV: 130636.4604  [HIGH sensitivity]  (min=123369.3750, max=140431.6719)
  block_6_QKV: 117532.4072  [HIGH sensitivity]  (min=109071.5391, max=128167.3984)
  block_7_QKV: 102728.7622  [HIGH sensitivity]  (min=95035.2812, max=111953.8672)
  block_8_QKV: 92190.3623  [HIGH sensitivity]  (min=84755.7031, max=100412.8281)
  block_9_QKV: 81044.0234  [HIGH sensitiv

In [18]:
# --- Budget sensitivity check: does the allocator actually differentiate blocks? ---
# At target_avg_bits=4.0, the budget (48 = 12 blocks x 4) lands exactly on a
# BIT_WIDTHS grid point. Below 4 bits the HAWQ-V2 perturbation term
# (||Q(W,bits)-W||_F^2) blows up steeply, so every block's "cheap" 16->8 and
# 8->4 downgrades get exhausted first regardless of its trace, and the loop
# stops the instant all 12 blocks hit the 4-bit floor -- before any block ever
# gets to compete for a downgrade below (or an upgrade above) that floor. That
# is why Section 10 above shows all 12 blocks at exactly 4 bits: it is not a
# broken allocator, it is a budget sitting exactly on that cliff.
#
# Sweeping a couple of nearby, non-grid-aligned budgets on the SAME scores
# confirms this: away from the cliff, the allocator cleanly separates
# high-trace (sensitive) blocks from low-trace ones.
print("Budget sensitivity check: reusing one JAB-score pass across a small sweep...")

# ===== CHANGED: attn_implementation="eager" -- this model goes through
# compute_all_jab_scores -> get_calibration_data, which now requests
# output_attentions=True for the KL target. Without eager, output_attentions
# silently comes back empty and target_attn_dict[idx] raises KeyError. =====
model_sweep_probe = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
model_sweep_probe.eval()

jab_scores_sweep = compute_all_jab_scores(model_sweep_probe, calibration_batches, DEVICE,
                                           n_embd, samples=20, n_batches_to_use=4)
allocator_input_sweep = scores_to_allocator_format_hawqv2(jab_scores_sweep, model_sweep_probe,
                                                           calibration_batches, DEVICE)

for avg_bits in [3.5, 4.3, 4.5, 6.0]:
    budget = avg_bits * len(allocator_input_sweep)
    alloc, cost_used, _ = greedy_allocate(allocator_input_sweep, budget)
    bits_used = sorted(set(alloc.values()))
    print(f"\navg_bits={avg_bits} (budget={budget:.1f}, cost_used={cost_used:.1f}):")
    print(f"  distinct bit-widths chosen: {bits_used}")
    for name, b in alloc.items():
        print(f"    {name}: {b} bits")

del model_sweep_probe

# --- Perplexity at each budget in the sweep ---
# Reuses allocator_input_sweep computed above; only the allocation + GPTQ + eval
# are redone per budget (each needs its own fresh, unquantized model).
ppl_by_budget = {}

for avg_bits in [3.5, 4.3, 4.5, 6.0]:
    budget = avg_bits * len(allocator_input_sweep)
    alloc, cost_used, _ = greedy_allocate(allocator_input_sweep, budget)

    print(f"\n=== avg_bits={avg_bits} (budget={budget:.1f}, cost_used={cost_used:.1f}) ===")
    model_budget = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
    model_budget.eval()
    model_budget = apply_allocation(model_budget, alloc, calibration_batches, DEVICE)

    ppl = evaluate_perplexity(model_budget, tokenizer)
    ppl_by_budget[avg_bits] = ppl
    print(f"Perplexity at avg_bits={avg_bits}: {ppl:.3f}")

    del model_budget

print("\n=== Perplexity vs. budget summary ===")
for avg_bits, ppl in ppl_by_budget.items():
    print(f"  avg_bits={avg_bits}: perplexity={ppl:.3f}")
print(f"  (uniform 4-bit baseline: {ppl_uniform:.3f})")


Budget sensitivity check: reusing one JAB-score pass across a small sweep...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing JAB scores for 12 blocks (averaged over 4 batches, 20 Hutchinson samples each, loss = MSE + 0.1*KL)...
  block_0_QKV: 7201.2488  [HIGH sensitivity]  (min=7033.9219, max=7452.7104)
  block_1_QKV: 18899.6523  [HIGH sensitivity]  (min=18035.9355, max=19458.3223)
  block_2_QKV: 114269.0938  [HIGH sensitivity]  (min=111497.3516, max=118226.8047)
  block_3_QKV: 298188.9844  [HIGH sensitivity]  (min=266085.4062, max=331194.7500)
  block_4_QKV: 139232.4805  [HIGH sensitivity]  (min=128641.9531, max=152150.5625)
  block_5_QKV: 130544.9414  [HIGH sensitivity]  (min=123943.2422, max=135999.4062)
  block_6_QKV: 121518.1406  [HIGH sensitivity]  (min=112884.7734, max=126192.5859)
  block_7_QKV: 105074.9453  [HIGH sensitivity]  (min=99269.7891, max=113028.2656)
  block_8_QKV: 93275.0605  [HIGH sensitivity]  (min=92104.5547, max=95458.3828)
  block_9_QKV: 82388.0000  [HIGH sensitivity]  (min=79618.2031, max=86280.8906)
  block_10_QKV: 73944.7715  [HIGH sensitivity]  (min=71617.9453, max=775

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 3 bits
  Block 1: 3 bits
  Block 2: 4 bits
  Block 3: 4 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 3 bits
  Block 9: 3 bits
  Block 10: 3 bits
  Block 11: 3 bits
Allocation done.
Perplexity at avg_bits=3.5: 28.522

=== avg_bits=4.3 (budget=51.6, cost_used=51.0) ===


...


Applying allocation
  Block 0: 3 bits
  Block 1: 4 bits
  Block 2: 4 bits
  Block 3: 8 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=4.3: 26.514

=== avg_bits=4.5 (budget=54.0, cost_used=52.0) ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 4 bits
  Block 3: 8 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=4.5: 26.204

=== avg_bits=6.0 (budget=72.0, cost_used=72.0) ===


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 8 bits
  Block 3: 8 bits
  Block 4: 8 bits
  Block 5: 8 bits
  Block 6: 8 bits
  Block 7: 8 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Perplexity at avg_bits=6.0: 25.046

=== Perplexity vs. budget summary ===
  avg_bits=3.5: perplexity=28.522
  avg_bits=4.3: perplexity=26.514
  avg_bits=4.5: perplexity=26.204
  avg_bits=6.0: perplexity=25.046
  (uniform 4-bit baseline: 26.225)


## 11. Joint attention-aware fine-tuning (fills in Untitled10's `joint_calibration_skeleton`)

Sections 8-10 only used `attention_loss` as an importance **score** (the JAB-Hessian trace) to decide *how many bits* each block gets -- the actual quantized weights were still produced by GPTQ minimizing the conventional `||XW - XW_hat||^2`, not the attention-output loss the PDF's Objective 1 asks for.

This section closes that gap: it fills in the interface Untitled10.ipynb sketches in `joint_calibration_skeleton` ("Week 2 task -- once JAB-Hessian is wired in"), using this notebook's own validated `attention_loss`/`compute_attention`. Each block's GPTQ-initialized `W_Q, W_K, W_V` are wrapped in a straight-through quantizer and fine-tuned with Adam to directly minimize `L(A(X), A_hat(X))`, jointly -- i.e. `min_{W_hat_Q,W_hat_K,W_hat_V} L(A(X), A_hat(X))` from the PDF, not a proxy for it.

**Correctness note:** `target_A` is always captured from a separate, untouched full-precision `model_float` -- never from `model_joint`, which is being progressively quantized. Distilling a quantized model toward its own (already-degraded) output would silently defeat the objective.

---

### Bug found after running this section: perplexity got *worse* (26.225 -> 32.660)

**Root cause.** `STEQuantize`'s fake-quantization grid is controlled by a `scale`
tensor, and the fine-tuning loop below used to build that scale with
`per_row_scale_flat`, which computes **one scale per input dimension, shared
across all 768 output channels, with no grouping at all**. But the GPTQ warm
start it fine-tunes from (`gptq_quantize_layer(..., group_size=128,
act_order=True)`) quantizes with **one scale per output channel, per group of
128 input columns** -- a completely different axis and granularity.

So the very first call to `ste_quantize(w_flat, scale_flat, bits)` inside the
fine-tuning loop -- *before Adam ever takes a step* -- silently re-quantized
the carefully-optimized GPTQ solution onto a coarser, misaligned grid. A
synthetic test with the exact same shapes/settings confirms this introduces
~9-10% relative weight error immediately at step 0 (see the sanity-check cell
below). Only 8 tiny (`lr=1e-4`) Adam steps per block can't repair that kind of
damage, and because blocks are quantized *sequentially* (block *i*'s
calibration input `X` already reflects blocks `0..i-1`'s quantization error),
the damage compounds -- consistent with block 0's fine-tuned loss (0.0008)
being ~56x smaller than block 11's (0.043) in the original run.

**Fix**, implemented below:
1. `gptq_quantize_layer` gained an optional `return_scale=True` mode that
   hands back the *exact* per-(output-channel, group) scale it used, in the
   original column order. `joint_attention_aware_finetune` now does its own
   GPTQ warm start (instead of a separate pre-quantization pass) and reuses
   that exact scale, so `ste_quantize` reproduces the GPTQ solution exactly
   at step 0 -- fine-tuning can now only move *away* from a correct starting
   point, never a silently-corrupted one.
2. **Best-iterate tracking + a safety net**: each block keeps the
   lowest-loss iterate seen during its (short, noisy) Adam trajectory, and
   only commits it if it actually beats the GPTQ-only starting loss;
   otherwise the block keeps its GPTQ-only weights. This gives a formal
   per-block guarantee that joint fine-tuning cannot make the *local*
   attention-reconstruction objective worse than plain GPTQ.
3. Gradient clipping, and calibration batches that rotate across the
   calibration set instead of every block reusing the same first 8 batches.


In [19]:
# --- Sanity check: does the fine-tuning scale match the GPTQ warm-start scale? ---
# Synthetic stand-in for one block's c_attn, small enough to run instantly,
# with the SAME shapes/settings pattern (group_size, act_order, bits) as the
# real pipeline. Demonstrates the bug (OLD per_row_scale_flat) and the fix
# (NEW: reuse gptq_quantize_layer's own return_scale) side by side.

def _demo_per_row_scale_flat_OLD(W0, qmax):
    """The buggy scale used by the original Section 11: one scale per INPUT
    row, shared across every output column, no grouping at all."""
    row_max = W0.detach().abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
    return (row_max / qmax).expand_as(W0).reshape(-1)

torch.manual_seed(0)
demo_n_embd, demo_group_size, demo_bits = 64, 32, 4  # small stand-ins for 768 / 128 / 4

W_demo = torch.randn(demo_n_embd, 3 * demo_n_embd) * 0.05
X_demo = torch.randn(2000, demo_n_embd)
H_demo = (2.0 * X_demo.T @ X_demo / X_demo.shape[0]).to(torch.float64)

W_after, scale_correct = gptq_quantize_layer(
    W_demo.clone(), H_demo, bits=demo_bits, group_size=demo_group_size,
    act_order=True, return_scale=True,
)
W_Q0, W_K0, W_V0 = W_after.split(demo_n_embd, dim=1)
w_flat_demo = torch.cat([W_Q0.flatten(), W_K0.flatten(), W_V0.flatten()])
qmax_demo = 2 ** (demo_bits - 1) - 1

# OLD (buggy): re-quantizing the GPTQ solution with the wrong-axis scale
scale_old = torch.cat([_demo_per_row_scale_flat_OLD(w, qmax_demo) for w in (W_Q0, W_K0, W_V0)])
w_old = torch.clamp(torch.round(w_flat_demo / scale_old), -qmax_demo, qmax_demo) * scale_old
err_old = ((w_old - w_flat_demo).norm() / w_flat_demo.norm()).item()

# NEW (fixed): reusing GPTQ's own exact scale
scale_Q, scale_K, scale_V = scale_correct.split(demo_n_embd, dim=1)
scale_new = torch.cat([scale_Q.flatten(), scale_K.flatten(), scale_V.flatten()])
w_new = torch.clamp(torch.round(w_flat_demo / scale_new), -qmax_demo, qmax_demo) * scale_new
err_new = ((w_new - w_flat_demo).norm() / w_flat_demo.norm()).item()

print(f"Relative error re-quantizing the GPTQ solution with the OLD scale : {err_old:.4%}")
print(f"Relative error re-quantizing the GPTQ solution with the NEW scale : {err_new:.6%}")
if err_old > 0.01 and err_new < 1e-4:
    print("\nConfirmed: the OLD scale silently corrupts the GPTQ warm start; the NEW scale is exact.")


Relative error re-quantizing the GPTQ solution with the OLD scale : 9.5696%
Relative error re-quantizing the GPTQ solution with the NEW scale : 0.000003%

Confirmed: the OLD scale silently corrupts the GPTQ warm start; the NEW scale is exact.


In [20]:
class STEQuantize(torch.autograd.Function):
    """
    Straight-through estimator for fake quantization: rounds onto the bit-grid
    in the forward pass (so downstream loss "sees" quantization error), but
    passes the incoming gradient through unchanged in the backward pass, so
    Adam can adjust the underlying float weights to compensate.
    """
    @staticmethod
    def forward(ctx, w, scale, bits):
        qmax = 2 ** (bits - 1) - 1
        q = torch.clamp(torch.round(w / scale), -qmax, qmax)
        return q * scale

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output, None, None


def ste_quantize(w, scale, bits):
    return STEQuantize.apply(w, scale, bits)


def joint_attention_aware_finetune(model_joint, model_float, calibration_batches, device,
                                    n_embd, bits=4, lr=1e-4, steps_per_block=8,
                                    group_size=128, damp_percent=0.01,
                                    grad_clip_norm=1.0, lambda_kl=0.1, verbose=True):
    """
    Objective 1 (`min_{W_hat_Q,W_hat_K,W_hat_V} L(A(X), A_hat(X))`), made robust:

      1. GPTQ-warm-starts each block INSIDE this function (instead of a
         separate pre-quantization pass) so we can capture the EXACT
         per-(output-channel, group) scale GPTQ used via `return_scale=True`.
         This is the fix for the bug described in the markdown above: the
         old code re-derived a scale with a different granularity/axis than
         GPTQ's own scale, silently corrupting the warm start before any
         fine-tuning step ran.
      2. Tracks the BEST iterate seen during each block's fine-tuning (by
         attention_loss on that block's calibration batches), not just the
         last one -- an ~8-step Adam trajectory is noisy.
      3. Safety net: a block's fine-tuned weights are committed only if they
         beat the GPTQ-only starting loss; otherwise the block keeps its
         GPTQ-only weights. Formally guarantees
         attention_loss(final) <= attention_loss(GPTQ-only warm start)
         for every block (a guarantee on the local reconstruction objective,
         which is itself a proxy for perplexity -- not a hard guarantee on
         perplexity -- but it removes this specific failure mode).
      4. Gradient clipping, and calibration batches that rotate across the
         calibration set per block instead of every block reusing the same
         first `steps_per_block` batches.

    `target_A` always comes from `model_float` (untouched, full precision).
    `X` (this block's input activations) comes from `model_joint`, so later
    blocks train against realistic post-quantization-error inputs from
    earlier blocks, consistent with how GPTQ itself behaves sequentially.

    # ===== CHANGED (KL divergence) =====
    # `target_attn` (also captured from model_float, alongside target_A) is
    # now threaded into every attention_loss call below with weight
    # `lambda_kl`, so the fine-tuning objective is MSE + lambda_kl*KL instead
    # of MSE-only. Pass lambda_kl=0.0 to reproduce the previous behavior.
    """
    n_blocks = len(model_joint.transformer.h)

    for block_idx in range(n_blocks):
        block = model_joint.transformer.h[block_idx]
        c_attn = block.attn.c_attn

        # --- GPTQ warm start, done HERE so we can capture its exact scale ---
        H = collect_hessian_via_hook(model_joint, c_attn, calibration_batches, device)
        _, scale_full = gptq_quantize_layer(
            c_attn.weight.data, H, bits=bits, group_size=group_size,
            damp_percent=damp_percent, act_order=True, return_scale=True,
        )  # c_attn.weight.data is now GPTQ-quantized in place

        W = c_attn.weight.data
        W_Q0, W_K0, W_V0 = W.split(n_embd, dim=1)
        w_flat = torch.cat([W_Q0.flatten(), W_K0.flatten(), W_V0.flatten()]).clone()
        w_flat.requires_grad_(True)

        # the EXACT scale GPTQ used -- ste_quantize(w_flat, scale_flat, bits)
        # reproduces w_flat exactly right now, before any Adam step.
        scale_Q, scale_K, scale_V = scale_full.split(n_embd, dim=1)
        scale_flat = torch.cat([scale_Q.flatten(), scale_K.flatten(), scale_V.flatten()])

        bias = c_attn.bias.data
        b_Q, b_K, b_V = bias.split(n_embd)

        optimizer = torch.optim.Adam([w_flat], lr=lr)

        # rotate through the calibration set instead of every block reusing
        # the same first `steps_per_block` batches
        start = (block_idx * steps_per_block) % len(calibration_batches)
        idxs = [(start + s) % len(calibration_batches) for s in range(steps_per_block)]
        batches = [calibration_batches[i] for i in idxs]

        with torch.no_grad():
            init_losses = []
            for batch in batches:
                # ===== CHANGED: target_attn is no longer discarded (was `_`) =====
                _, target_A, target_attn = get_calibration_data_for_block(model_float, batch, device, block_idx)
                X, _, _ = get_calibration_data_for_block(model_joint, batch, device, block_idx)
                # ===== CHANGED: target_attn/lambda_kl now passed through =====
                init_losses.append(attention_loss(w_flat.detach(), X, target_A, n_embd,
                                                    target_attn=target_attn, lambda_kl=lambda_kl,
                                                    b_Q=b_Q, b_K=b_K, b_V=b_V).item())
            init_loss = sum(init_losses) / len(init_losses)

        best_loss = init_loss
        best_w = w_flat.detach().clone()

        for batch in batches:
            # ===== CHANGED: target_attn is no longer discarded (was `_`) =====
            _, target_A, target_attn = get_calibration_data_for_block(model_float, batch, device, block_idx)
            X, _, _ = get_calibration_data_for_block(model_joint, batch, device, block_idx)

            optimizer.zero_grad()
            w_q = ste_quantize(w_flat, scale_flat, bits)
            # ===== CHANGED: target_attn/lambda_kl now passed through, so this
            # is the actual training objective, not just the sanity-check one =====
            loss = attention_loss(w_q, X, target_A, n_embd, target_attn=target_attn,
                                   lambda_kl=lambda_kl, b_Q=b_Q, b_K=b_K, b_V=b_V)
            loss.backward()
            torch.nn.utils.clip_grad_norm_([w_flat], grad_clip_norm)
            optimizer.step()

            loss_val = loss.item()
            if loss_val < best_loss:
                best_loss = loss_val
                with torch.no_grad():
                    best_w = ste_quantize(w_flat, scale_flat, bits).detach().clone()

        if best_loss <= init_loss:
            W_Q, W_K, W_V = reshape_weights(best_w, n_embd)
            c_attn.weight.data.copy_(torch.cat([W_Q, W_K, W_V], dim=1))
            status = f"improved {init_loss:.6f} -> {best_loss:.6f}"
        else:
            # best_w never beat the GPTQ-only starting point; c_attn.weight.data
            # is already that GPTQ-only solution (nothing else has touched it),
            # so we simply don't overwrite it with the worse fine-tuned weights.
            status = f"kept GPTQ-only ({init_loss:.6f}; fine-tune best was {best_loss:.6f})"

        if verbose:
            print(f"  Block {block_idx}: {status}")

    return model_joint

In [21]:
print("Loading a fresh full-precision GPT-2 as the fixed reference (target_A source)...")
# ===== CHANGED: attn_implementation="eager" -- model_float is the source of
# target_attn (the KL target) via get_calibration_data_for_block. =====
model_float = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
model_float.eval()

print("Loading a second, still full-precision GPT-2 for joint attention-aware fine-tuning...")
print("(GPTQ warm-start now happens INSIDE joint_attention_aware_finetune, per block,")
print(" so the exact scale it used can be reused for fine-tuning -- see the fix above.)")
# ===== CHANGED: attn_implementation="eager" -- model_joint also goes through
# get_calibration_data_for_block (for X), which now requests
# output_attentions=True internally. =====
model_joint = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
model_joint.eval()

print("\nRunning joint attention-aware fine-tuning (Objective 1: minimizes L(A(X), A_hat(X)) directly)...")
# ===== CHANGED: lambda_kl=0.1 -- fine-tuning now optimizes MSE + 0.1*KL,
# not MSE alone. Set lambda_kl=0.0 to reproduce the previous MSE-only run. =====
model_joint = joint_attention_aware_finetune(model_joint, model_float, calibration_batches, DEVICE,
                                              n_embd, bits=4, lr=1e-4, steps_per_block=8,
                                              group_size=128, lambda_kl=0.1)

print("\nEvaluating perplexity...")
ppl_joint = evaluate_perplexity(model_joint, tokenizer)
print(f"\nJoint attention-aware fine-tuned (Objective 1) perplexity: {ppl_joint:.3f}")

Loading a fresh full-precision GPT-2 as the fixed reference (target_A source)...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading a second, still full-precision GPT-2 for joint attention-aware fine-tuning...
(GPTQ warm-start now happens INSIDE joint_attention_aware_finetune, per block,
 so the exact scale it used can be reused for fine-tuning -- see the fix above.)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Running joint attention-aware fine-tuning (Objective 1: minimizes L(A(X), A_hat(X)) directly)...
  Block 0: improved 5.405362 -> 4.992214
  Block 1: improved 6.063256 -> 5.784966
  Block 2: improved 23.224470 -> 20.818268
  Block 3: improved 67.033885 -> 64.145142
  Block 4: improved 40.427207 -> 38.103489
  Block 5: improved 42.895506 -> 39.759853
  Block 6: improved 56.008645 -> 50.820572
  Block 7: improved 67.789245 -> 62.269867
  Block 8: improved 64.470241 -> 57.649887
  Block 9: improved 65.675274 -> 61.888138
  Block 10: improved 41.896248 -> 39.634563
  Block 11: improved 31.654732 -> 27.653318

Evaluating perplexity...

Joint attention-aware fine-tuned (Objective 1) perplexity: 26.233


## 12. Compare results

In [22]:
print(f"Uniform  4-bit GPTQ perplexity              : {ppl_uniform:.3f}")
print(f"Adaptive JAB-Hessian perplexity              : {ppl_adaptive:.3f}")
print(f"Joint attention-aware fine-tuned perplexity  : {ppl_joint:.3f}")
print(f"Adaptive vs. uniform difference               : {ppl_adaptive - ppl_uniform:+.3f}")
print(f"Joint    vs. uniform difference               : {ppl_joint - ppl_uniform:+.3f}")


Uniform  4-bit GPTQ perplexity              : 26.225
Adaptive JAB-Hessian perplexity              : 26.478
Joint attention-aware fine-tuned perplexity  : 26.233
Adaptive vs. uniform difference               : +0.253
Joint    vs. uniform difference               : +0.007


## 13. Full pipeline: Fisher‑coupling adaptive allocation

In [23]:
"""
fisher_coupling.py (embedded cell)
Measures actual Q/K/V interaction via Fisher information (gradient dot products).
Unlike Hessian trace, this CAN see cross-matrix coupling.
Requires: get_calibration_data() has been updated with 'with_grad' parameter.
"""

def compute_fisher_coupling(model, block_idx, X, target_A, target_attn, n_embd, b_Q, b_K, b_V, use_kl=True, lambda_kl=0.1, probe_bits=2):
    """
    probe_bits: the gradient MUST be evaluated at a PERTURBED (quantized)
    point, not the exact full-precision weights -- at full precision,
    A_hat == target_A exactly, the MSE residual is exactly zero, and the
    gradient (which is proportional to the residual) collapses to exactly
    zero regardless of the true Jacobian. Quantizing first gives a real,
    nonzero residual to differentiate against.
    """
    block = model.transformer.h[block_idx]
    W = block.attn.c_attn.weight.data  # (768, 2304) = [W_Q | W_K | W_V]

    # Simple round-to-nearest fake-quantization as the probe point (cheap,
    # no Hessian needed here -- just needs A_hat != target_A)
    qmax = 2 ** (probe_bits - 1) - 1
    scale = (W.abs().amax(dim=0, keepdim=True) / qmax).clamp(min=1e-8)
    W_quantized = torch.clamp(torch.round(W / scale), -qmax, qmax) * scale

    W_Q, W_K, W_V = [w.clone().requires_grad_(True) for w in W_quantized.split(n_embd, dim=1)]
    w_flat = torch.cat([W_Q.flatten(), W_K.flatten(), W_V.flatten()])

    loss = attention_loss(
        w_flat, X, target_A, n_embd,
        target_attn=target_attn if use_kl else None,
        lambda_kl=lambda_kl if use_kl else 0.0,
        b_Q=b_Q, b_K=b_K, b_V=b_V
    )
    g_Q, g_K, g_V = torch.autograd.grad(loss, [W_Q, W_K, W_V])
    g_Q, g_K, g_V = g_Q.flatten(), g_K.flatten(), g_V.flatten()

    def cos(a, b):
        return (torch.dot(a, b) / (a.norm() * b.norm() + 1e-12)).item()

    return {
        "QQ": torch.dot(g_Q, g_Q).item(), "KK": torch.dot(g_K, g_K).item(), "VV": torch.dot(g_V, g_V).item(),
        "QK_cos": cos(g_Q, g_K), "QV_cos": cos(g_Q, g_V), "KV_cos": cos(g_K, g_V),
    }


def compute_all_fisher_coupling(model, batches, device, n_embd, n_batches_to_use=8, use_kl=True, lambda_kl=0.1, probe_bits=2):
    n_blocks = len(model.transformer.h)
    use_batches = batches[:n_batches_to_use]

    raw = {f"block_{i}_QKV": [] for i in range(n_blocks)}  # list of per-batch dicts, per block

    print(f"\nComputing Fisher coupling for {n_blocks} blocks over {len(use_batches)} batches...")

    for b in use_batches:
        X_dict, target_A_dict, target_attn_dict = get_calibration_data(model, b, device, with_grad=True)
        for idx in range(n_blocks):
            block = model.transformer.h[idx]
            bias = block.attn.c_attn.bias.data
            b_Q, b_K, b_V = bias.split(n_embd)

            result = compute_fisher_coupling(
                model, idx, X_dict[idx], target_A_dict[idx], target_attn_dict[idx],
                n_embd, b_Q, b_K, b_V, use_kl=use_kl, lambda_kl=lambda_kl, probe_bits=2
            )
            raw[f"block_{idx}_QKV"].append(result)

    coupling_scores = {}
    for block_name, per_batch in raw.items():
        avg_QQ = sum(d["QQ"] for d in per_batch) / len(per_batch)
        avg_KK = sum(d["KK"] for d in per_batch) / len(per_batch)
        avg_VV = sum(d["VV"] for d in per_batch) / len(per_batch)

        signed_QK = sum(d["QK_cos"] for d in per_batch) / len(per_batch)
        signed_QV = sum(d["QV_cos"] for d in per_batch) / len(per_batch)
        signed_KV = sum(d["KV_cos"] for d in per_batch) / len(per_batch)

        magnitude_QK = sum(abs(d["QK_cos"]) for d in per_batch) / len(per_batch)
        magnitude_QV = sum(abs(d["QV_cos"]) for d in per_batch) / len(per_batch)
        magnitude_KV = sum(abs(d["KV_cos"]) for d in per_batch) / len(per_batch)

        coupling_scores[block_name] = {
            "signed": {"QK": signed_QK, "QV": signed_QV, "KV": signed_KV},
            "magnitude": {"QK": magnitude_QK, "QV": magnitude_QV, "KV": magnitude_KV},
            "raw": {"QQ": avg_QQ, "KK": avg_KK, "VV": avg_VV},
        }

        print(f"  {block_name}: diag(QQ={avg_QQ:.4f}, KK={avg_KK:.4f}, VV={avg_VV:.4f})  "
              f"signed(QK={signed_QK:+.4f}, QV={signed_QV:+.4f}, KV={signed_KV:+.4f})  "
              f"|magnitude|(QK={magnitude_QK:.4f}, QV={magnitude_QV:.4f}, KV={magnitude_KV:.4f})")

    print("Done!\n")
    return coupling_scores


def fisher_scores_to_allocator_format(fisher_scores, model, calibration_batches, device, bit_widths=None):
    """
    Convert per-block Fisher scores (diagonal sums) to HAWQ-V2 style
    allocator input: Omega_i(bits) = (Fisher_diag_sum_i) * ||Q(W_i)-W_i||^2.
    """
    if bit_widths is None:
        bit_widths = [2, 3, 4, 8, 16]

    allocator_input = {}
    for block_name, scores in fisher_scores.items():
        block_idx = int(block_name.split("_")[1])
        block = model.transformer.h[block_idx]
        c_attn = block.attn.c_attn
        original_W = c_attn.weight.data

        # Collect Hessian H = 2 X^T X (same as for JAB)
        H = collect_hessian_via_hook(model, c_attn, calibration_batches, device)

        # Scalar sensitivity: sum of diagonal Fisher entries (QQ + KK + VV)
        scalar_sensitivity = scores["raw"]["QQ"] + scores["raw"]["KK"] + scores["raw"]["VV"]

        sensitivity_by_bits = {}
        for bits in bit_widths:
            perturbation = measure_weight_perturbation(original_W, H, bits)
            sensitivity_by_bits[bits] = scalar_sensitivity * perturbation

        allocator_input[block_name] = sensitivity_by_bits
        print(f"  {block_name}: " + ", ".join(f"{b}bit={v:.4e}" for b, v in sensitivity_by_bits.items()))

    return allocator_input

In [24]:
print("\n" + "=" * 70)
print("FISHER-COUPLING ADAPTIVE ALLOCATION: MSE + KL")
print("=" * 70)

BUDGETS = [3.5, 4.3, 4.5, 6.0]
N_BATCHES_FOR_SCORING = 8

results = {"MSE + KL": {}}

for use_kl, loss_name in [(True, "MSE + KL")]:  # only KL
    print(f"\n{'='*70}")
    print(f"Computing Fisher scores with {loss_name} loss...")
    print("=" * 70)

    model_fisher = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
    model_fisher.eval()

    torch.manual_seed(42)
    fisher_scores = compute_all_fisher_coupling(
        model=model_fisher,
        batches=calibration_batches,
        device=DEVICE,
        n_embd=n_embd,
        n_batches_to_use=N_BATCHES_FOR_SCORING,
        use_kl=use_kl,
        lambda_kl=0.1   # or 0.01, adjust here
    )

    print("\nConverting Fisher scores to allocator format...")
    allocator_input_fisher = fisher_scores_to_allocator_format(
        fisher_scores, model_fisher, calibration_batches, DEVICE
    )

    n_blocks = len(allocator_input_fisher)

    for target_avg_bits in BUDGETS:
        print(f"\n{'-'*60}")
        print(f"Testing {loss_name}: avg_bits = {target_avg_bits}")
        print("-" * 60)

        budget = target_avg_bits * n_blocks
        assignment, cost_used, sensitivity = greedy_allocate(allocator_input_fisher, budget)
        print(f"Total cost: {cost_used:.1f} bits | Average bits: {cost_used / n_blocks:.2f}")
        print("Per-block allocation:")
        for block_name, bits in assignment.items():
            print(f"    {block_name}: {bits} bits")

        print("Loading fresh model for quantization...")
        model_budget = AutoModelForCausalLM.from_pretrained("gpt2", attn_implementation="eager").to(DEVICE)
        model_budget.eval()

        print("Applying the allocation...")
        model_budget = apply_allocation(model_budget, assignment, calibration_batches, DEVICE)

        print("Evaluating perplexity...")
        ppl = evaluate_perplexity(model_budget, tokenizer)
        results[loss_name][target_avg_bits] = ppl
        print(f"\n{loss_name} adaptive {target_avg_bits}-bit perplexity: {ppl:.3f}")

        del model_budget
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    del model_fisher
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# --- Final Table (now only one column) ---
print("\n" + "=" * 70)
print("FINAL RESULTS: FISHER-COUPLING ADAPTIVE ALLOCATION (MSE + KL)")
print("=" * 70)
print(f"\n{'Budget':>8} | {'Perplexity':>12}")
print("-" * 25)
for b in BUDGETS:
    ppl = results["MSE + KL"][b]
    print(f"{b:>8.1f} | {ppl:>12.3f}")


FISHER-COUPLING ADAPTIVE ALLOCATION: MSE + KL

Computing Fisher scores with MSE + KL loss...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Computing Fisher coupling for 12 blocks over 8 batches...
  block_0_QKV: diag(QQ=2986.2972, KK=2032.4079, VV=0.0000)  signed(QK=-0.0058, QV=-0.0078, KV=-0.0025)  |magnitude|(QK=0.0067, QV=0.0083, KV=0.0035)
  block_1_QKV: diag(QQ=73845.5107, KK=15589.6246, VV=0.0005)  signed(QK=-0.1048, QV=+0.0081, KV=-0.0003)  |magnitude|(QK=0.1048, QV=0.0081, KV=0.0019)
  block_2_QKV: diag(QQ=3144283.9688, KK=219231.4160, VV=0.0018)  signed(QK=-0.1764, QV=+0.0098, KV=-0.0055)  |magnitude|(QK=0.1764, QV=0.0098, KV=0.0055)
  block_3_QKV: diag(QQ=3165183.4375, KK=327056.6914, VV=0.0017)  signed(QK=-0.1272, QV=+0.0034, KV=-0.0212)  |magnitude|(QK=0.1272, QV=0.0083, KV=0.0215)
  block_4_QKV: diag(QQ=1093973.1875, KK=222054.2949, VV=0.0012)  signed(QK=-0.0823, QV=-0.0070, KV=-0.0046)  |magnitude|(QK=0.0823, QV=0.0100, KV=0.0107)
  block_5_QKV: diag(QQ=486682.3789, KK=292307.8633, VV=0.0072)  signed(QK=-0.1517, QV=-0.0017, KV=+0.0685)  |magnitude|(QK=0.1517, QV=0.0142, KV=0.0685)
  block_6_QKV: diag(QQ=311

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Applying the allocation...

Applying allocation
  Block 0: 2 bits
  Block 1: 3 bits
  Block 2: 4 bits
  Block 3: 4 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 3 bits
  Block 10: 3 bits
  Block 11: 3 bits
Allocation done.
Evaluating perplexity...

MSE + KL adaptive 3.5-bit perplexity: 46.918

------------------------------------------------------------
Testing MSE + KL: avg_bits = 4.3
------------------------------------------------------------
Total cost: 51.0 bits | Average bits: 4.25
Per-block allocation:
    block_0_QKV: 2 bits
    block_1_QKV: 3 bits
    block_2_QKV: 8 bits
    block_3_QKV: 8 bits
    block_4_QKV: 4 bits
    block_5_QKV: 4 bits
    block_6_QKV: 4 bits
    block_7_QKV: 4 bits
    block_8_QKV: 4 bits
    block_9_QKV: 4 bits
    block_10_QKV: 3 bits
    block_11_QKV: 3 bits
Loading fresh model for quantization...


...

Applying the allocation...

Applying allocation
  Block 0: 2 bits
  Block 1: 3 bits
  Block 2: 8 bits
  Block 3: 8 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 3 bits
  Block 11: 3 bits
Allocation done.
Evaluating perplexity...

MSE + KL adaptive 4.3-bit perplexity: 45.768

------------------------------------------------------------
Testing MSE + KL: avg_bits = 4.5
------------------------------------------------------------
Total cost: 54.0 bits | Average bits: 4.50
Per-block allocation:
    block_0_QKV: 2 bits
    block_1_QKV: 4 bits
    block_2_QKV: 8 bits
    block_3_QKV: 8 bits
    block_4_QKV: 4 bits
    block_5_QKV: 4 bits
    block_6_QKV: 4 bits
    block_7_QKV: 4 bits
    block_8_QKV: 4 bits
    block_9_QKV: 4 bits
    block_10_QKV: 4 bits
    block_11_QKV: 4 bits
Loading fresh model for quantization...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Applying the allocation...

Applying allocation
  Block 0: 2 bits
  Block 1: 4 bits
  Block 2: 8 bits
  Block 3: 8 bits
  Block 4: 4 bits
  Block 5: 4 bits
  Block 6: 4 bits
  Block 7: 4 bits
  Block 8: 4 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Evaluating perplexity...

MSE + KL adaptive 4.5-bit perplexity: 39.734

------------------------------------------------------------
Testing MSE + KL: avg_bits = 6.0
------------------------------------------------------------
Total cost: 72.0 bits | Average bits: 6.00
Per-block allocation:
    block_0_QKV: 4 bits
    block_1_QKV: 4 bits
    block_2_QKV: 8 bits
    block_3_QKV: 8 bits
    block_4_QKV: 8 bits
    block_5_QKV: 8 bits
    block_6_QKV: 8 bits
    block_7_QKV: 4 bits
    block_8_QKV: 8 bits
    block_9_QKV: 4 bits
    block_10_QKV: 4 bits
    block_11_QKV: 4 bits
Loading fresh model for quantization...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Applying the allocation...

Applying allocation
  Block 0: 4 bits
  Block 1: 4 bits
  Block 2: 8 bits
  Block 3: 8 bits
  Block 4: 8 bits
  Block 5: 8 bits
  Block 6: 8 bits
  Block 7: 4 bits
  Block 8: 8 bits
  Block 9: 4 bits
  Block 10: 4 bits
  Block 11: 4 bits
Allocation done.
Evaluating perplexity...

MSE + KL adaptive 6.0-bit perplexity: 25.132

FINAL RESULTS: FISHER-COUPLING ADAPTIVE ALLOCATION (MSE + KL)

  Budget |   Perplexity
-------------------------
     3.5 |       46.918
     4.3 |       45.768
     4.5 |       39.734
     6.0 |       25.132


In [25]:
# ppl_uniform should already exist from the uniform baseline (float)
# ppl_by_budget should exist from the JAB sweep (dict: {bits: perplexity})
# ppl_fisher = results["MSE + KL"]   # extract from the Fisher run

ppl_fisher = results["MSE + KL"]

print("\n" + "=" * 60)
print("COMPARISON: JAB-Hessian vs Fisher-coupling")
print("=" * 60)

print(f"\n{'Budget':>8} | {'Uniform 4bit':>12} | {'JAB':>12} | {'Fisher':>12}")
print("-" * 60)

for b in [3.5, 4.3, 4.5, 6.0]:
    # Uniform baseline (same for all budgets as reference)
    uniform_val = f"{ppl_uniform:.3f}"   # show as a string for consistency

    # JAB results (use .get to avoid KeyError if a budget is missing)
    jab_val = ppl_by_budget.get(b, "N/A")

    # Fisher results
    fisher_val = ppl_fisher.get(b, "N/A")

    # Format numeric values to 3 decimals
    if isinstance(uniform_val, str):
        uniform_display = uniform_val
    else:
        uniform_display = f"{uniform_val:.3f}"
    if isinstance(jab_val, str):
        jab_display = jab_val
    else:
        jab_display = f"{jab_val:.3f}"
    if isinstance(fisher_val, str):
        fisher_display = fisher_val
    else:
        fisher_display = f"{fisher_val:.3f}"

    print(f"{b:>8.1f} | {uniform_display:>12} | {jab_display:>12} | {fisher_display:>12}")


COMPARISON: JAB-Hessian vs Fisher-coupling

  Budget | Uniform 4bit |          JAB |       Fisher
------------------------------------------------------------
     3.5 |       26.225 |       28.522 |       46.918
     4.3 |       26.225 |       26.514 |       45.768
     4.5 |       26.225 |       26.204 |       39.734
     6.0 |       26.225 |       25.046 |       25.132
